# 13 — Phiên 60 phút: dựng dữ liệu và tune lại trên validation

Người dùng duyệt ngày 24/11/2026: so sánh đầy đủ phiên **60 phút** với giao thức chính 30 phút, có tune lại siêu tham số. Giao thức chính (30 phút) và mọi artifact của notebook 01–12 **giữ nguyên**; kết quả 30 phút vẫn là số chính thức, 60 phút là báo cáo phụ.

- Dữ liệu: chạy lại tạo phiên với ngưỡng 3.600.000 ms, **cùng mốc chia** 0.8/0.9 của timestamp sự kiện sạch (không phụ thuộc ngưỡng phiên), chia nguyên phiên, phiên cắt qua mốc → `boundary_drop`. Mẫu prefix–target, vocabulary và `known_at` dựng lại đúng luật của notebook 04 (`known_at = min(sự kiện đầu tiên trong dữ liệu fit, bản thuộc tính đầu tiên)`).
- Tune trên validation 60 phút (mô hình fit trên train 60 phút), **cùng lưới và cùng thủ tục** với notebook gốc:
  - Popularity (lưới trọng số hành vi), S-POP, Repeat-last — như notebook 05;
  - ItemKNN, SKNN, V-SKNN, STAN, VSTAN, SR — đúng lưới 51 lần chạy của notebook 06;
  - GRU4Rec — cấu hình kiến trúc đã chọn ở notebook 07 (`bpr-max_official`), **chọn số epoch** bằng early stopping NDCG@20 (tối đa 8, patience 2);
  - SR-GNN — cấu hình đã chọn ở notebook 08, chọn số epoch (tối đa 6, patience 2);
  - Reranker kiểu HySAR — học lại trên validation 60 phút, CV 2 fold theo phiên, cùng lưới 2 cấu hình như notebook 09.
- Đầu ra: `data/processed/session60/` (sự kiện, mẫu, vocabulary, `known_at`), `models/session60/`, `reports/session60/validation.json`. Notebook 14 đọc các file này để chạy test.
- Theo dõi tiến độ: đặt biến môi trường `RR_PROGRESS_LOG` thành một đường dẫn tệp thì mỗi bước được ghi thêm một dòng vào đó.

## 1. Cấu hình và kiểm tra đầu vào

In [ ]:
from pathlib import Path
import itertools, os, sys, time

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import lightgbm as lgb
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, require, save_verified, utc_now, verify_inputs, write_json
from rr.ablation import build_samples, known_at_for, sessionize, split_summary
from rr.candidates import CandidateFilter
from rr.evaluation import flat, run
from rr.models import ItemKNN, SequentialRules, SessionKNN, popularity, repeat_last, spop
from rr.samples import build_vocab, prefix_events
from rr import gru as G
from rr import srgnn as S
from rr.rerank import add_features, candidate_table, feature_columns, label, ranked_lists, session_folds

PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
PROCESSED = ROOT / "data/processed"
DATA60 = PROCESSED / "session60"
OUT = ROOT / "reports/session60"
MODEL_DIR = ROOT / "models/session60"
THRESHOLD_MS = 60 * 60 * 1000
FALLBACK_TOP_N = PROTOCOL["candidates"]["fallback_top_n"]
MAX_GROUPS = PROTOCOL["knn"]["max_prefix_groups"]
MAX_LEN = PROTOCOL["long_sessions"]["gru_train_max_prefix_groups"] + 1
SEED = PROTOCOL["seed"]
WIDTH, TOP_N = 200, 100
GRU_MAX_EPOCHS, SRGNN_MAX_EPOCHS, PATIENCE = 8, 6, 2   # same procedure as notebooks 07 / 08
torch.set_num_threads(4)

LOG = os.environ.get("RR_PROGRESS_LOG")
def log(msg):
    line = f"{pd.Timestamp.now(tz='UTC'):%H:%M:%S} [13] {msg}"
    print(line, flush=True)
    if LOG:
        with open(LOG, "a", encoding="utf-8") as f:
            f.write(line + "\n")

prev = {n: read_json(ROOT / f"reports/{n}/metrics.json") for n in ["events_clean", "session_split", "metadata"]}
frozen30 = read_json(ROOT / "reports/final/frozen.json")["files"]
expected = {"data/processed/events_clean.parquet": prev["events_clean"]["output"],
            "data/processed/split_config.json": prev["session_split"]["split_config"]}
for p in ["item_property_history", "item_catalog"]:
    expected[f"data/processed/{p}.parquet"] = prev["metadata"]["artifacts"][f"data/processed/{p}.parquet"]
for p in ["configs/protocol.json", "models/gru4rec/config.json", "models/srgnn/config.json",
          *[f"reports/eval/{n}_validation.json" for n in ["popularity", "spop", "repeat_last", "knn", "gru4rec", "srgnn", "reranker"]]]:
    expected[p] = frozen30[p]
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
inputs["reports/data_ablation/metrics.json"] = fingerprint(ROOT / "reports/data_ablation/metrics.json")
print("Đầu vào khớp SHA-256:", len(inputs), "tệp")
print(environment(), "| torch", torch.__version__, "| lightgbm", lgb.__version__)
log("bắt đầu")

## 2. Ví dụ biên

- Ngưỡng 60 phút: khoảng nghỉ đúng 60 phút vẫn cùng phiên, 61 phút thì tách.
- `known_at` lấy sự kiện đầu tiên của **dữ liệu fit** (train, hoặc train + validation khi refit), không lấy từ tập đánh giá.
- Lưới trọng số hành vi có 16 cấu hình như notebook 05.

In [ ]:
MIN = 60_000
_e = pd.DataFrame({"event_id": [0, 1, 2], "timestamp": [0, 60 * MIN, 121 * MIN], "visitorid": [1, 1, 1], "itemid": [5, 6, 7],
                   "event": pd.Categorical(["view"] * 3, categories=["view", "addtocart", "transaction"])})
assert sessionize(_e, THRESHOLD_MS, (10**12, 2 * 10**12)).sort_values("event_id")["session_id"].tolist() == [1, 1, 2]
_cat = pd.DataFrame({"itemid": [5, 6], "first_property_ms": pd.array([pd.NA, 1], dtype="Int64")})
_fit = pd.DataFrame({"itemid": [5, 6], "timestamp": [100, 50]})
assert known_at_for(_cat, _fit, "k")["k"].tolist() == [100, 1]
_grid = PROTOCOL["behavior_weights"]["grid"]
assert len(list(itertools.product(_grid["view"], _grid["addtocart"], _grid["transaction"]))) == 16
del _e, _cat, _fit, _grid
print("Đã kiểm tra ví dụ biên.")

## 3. Dựng dữ liệu phiên 60 phút

Số phiên, số query theo tập phải khớp bảng mô tả của notebook 12 (mục C). Target giữ ID gốc kể cả ngoài vocabulary; `in_vocab` và `candidate_eligible` chỉ để báo cáo, không loại khỏi mẫu số.

In [ ]:
t0 = time.perf_counter()
clean = pd.read_parquet(PROCESSED / "events_clean.parquet", columns=["event_id", "timestamp", "visitorid", "event", "itemid"])
catalog = pd.read_parquet(PROCESSED / "item_catalog.parquet")
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
split_cfg = read_json(PROCESSED / "split_config.json")
CUTOFFS = split_cfg["cutoff_ms"]
events = sessionize(clean, THRESHOLD_MS, CUTOFFS)
del clean
SUMMARY = split_summary(events)
abl = read_json(ROOT / "reports/data_ablation/metrics.json")["C_session_threshold"]["describe_by_minutes"]["60"]
for sp in ["train", "validation", "test"]:
    require(SUMMARY[sp]["prefix_target_pairs"] == abl[f"{sp}_queries"], f"Số query {sp} khác notebook 12.")
require(SUMMARY["boundary_drop"]["sessions"] == abl["boundary_drop_sessions"], "boundary_drop khác notebook 12.")

train_ev = events[events["split"] == "train"]
val_ev = events[events["split"] == "validation"]
trainval_ev = events[events["split"].isin(["train", "validation"])]
vocab = build_vocab(train_ev)
vocab_tv = build_vocab(trainval_ev)
known = known_at_for(catalog, train_ev, "known_at_train_ms").merge(
    known_at_for(catalog, trainval_ev, "known_at_trainval_ms")[["itemid", "known_at_trainval_ms"]], on="itemid", validate="one_to_one")
flt = CandidateFilter.from_frames(vocab, known, history, "known_at_train_ms")

samples, data_stats, next_id = {}, {}, 0
for sp in ["train", "validation", "test"]:
    q, t = build_samples(events, sp, first_query_id=next_id)
    next_id += len(q)
    tt = t.merge(q[["query_id", "query_time_ms", "target_time_ms"]], on="query_id", how="left", validate="many_to_one")
    require((tt["target_time_ms"] > tt["query_time_ms"]).all(), f"{sp}: target không nằm sau query.")
    t["in_vocab"] = t["itemid"].isin(vocab["itemid"]).to_numpy()
    t["candidate_eligible"] = flt.eligible(tt["itemid"].to_numpy(), tt["query_time_ms"].to_numpy())
    flags = t.groupby("query_id").agg(n_targets=("itemid", "size"), any_in_vocab=("in_vocab", "any"),
                                      any_repeat=("target_is_repeat", "any"), all_repeat=("target_is_repeat", "all"),
                                      target_has_strong=("has_strong_behavior", "any"))
    q = q.join(flags, on="query_id")
    q["n_targets"] = q["n_targets"].astype(np.int32)
    samples[sp] = (q, t)
    data_stats[sp] = {"queries": int(len(q)), "target_occurrences": int(len(t)), "oov_target_pct": 100 * float((~t["in_vocab"]).mean()),
                      "not_candidate_eligible_pct": 100 * float((~t["candidate_eligible"]).mean()),
                      "repeat_target_pct": 100 * float(t["target_is_repeat"].mean()),
                      "strong_behavior_target_pct": 100 * float(t["has_strong_behavior"].mean()),
                      "queries_all_oov": int((~q["any_in_vocab"]).sum())}
    require(len(q) == SUMMARY[sp]["prefix_target_pairs"], f"{sp}: số query khác số cặp prefix-target.")
require(train_ev["timestamp"].max() < samples["validation"][0]["query_time_ms"].min(), "Train phải nằm trước validation.")
require(trainval_ev["timestamp"].max() < samples["test"][0]["query_time_ms"].min(), "Train + validation phải nằm trước test.")
display(pd.DataFrame(data_stats).T)
display(pd.DataFrame(SUMMARY).T)

artifacts = {}
for frame, name, key in [(events, "events_sessionized.parquet", ["event_id"]), (vocab, "item_vocab.parquet", ["itemid"]),
                         (vocab_tv, "item_vocab_trainval.parquet", ["itemid"]),
                         (known[["itemid", "known_at_train_ms", "known_at_trainval_ms"]], "item_known_at.parquet", ["itemid"])]:
    info = save_verified(frame.reset_index(drop=True), DATA60 / name, key, ROOT); artifacts[info.pop("path")] = info
for sp, (q, t) in samples.items():
    for frame, kind, key in [(q, "queries", ["query_id"]), (t, "targets", ["query_id", "itemid"])]:
        info = save_verified(frame.reset_index(drop=True), DATA60 / f"samples/{sp}_{kind}.parquet", key, ROOT)
        artifacts[info.pop("path")] = info
log(f"dữ liệu 60 phút xong ({time.perf_counter() - t0:.0f}s): " + ", ".join(f"{sp} {s['queries']:,} query" for sp, s in data_stats.items()))

## 4. Popularity, S-POP, Repeat-last (như notebook 05)

In [ ]:
val_q, val_t = samples["validation"]
qids, qtimes = val_q["query_id"].to_numpy(), val_q["query_time_ms"].to_numpy()
EMPTY = np.empty((len(val_q), 0), dtype=np.int64)
grid = PROTOCOL["behavior_weights"]["grid"]
pop_reports, pop_rank = {}, {}
for v, c, b in itertools.product(grid["view"], grid["addtocart"], grid["transaction"]):
    w = dict(view=v, addtocart=c, transaction=b)
    name = f"pop_v{v}_c{c}_b{b}"
    items, scores = popularity(train_ev, w)
    report, *_ = run(EMPTY, None, val_q, val_t, flt, items[:FALLBACK_TOP_N], scores[:FALLBACK_TOP_N])
    require(report["short_lists"] == 0, f"{name}: có query thiếu item sau lọc.")
    report["weights"] = w
    pop_reports[name], pop_rank[name] = report, (items, scores)
UNIFORM = "pop_v1_c1_b1"
best_pop = flat(pop_reports).sort_values("ndcg@20", ascending=False).index[0]
fallback_name = best_pop if pop_reports[best_pop]["all"]["ndcg@20"] > pop_reports[UNIFORM]["all"]["ndcg@20"] else UNIFORM
fb_items, fb_scores = (a[:FALLBACK_TOP_N] for a in pop_rank[fallback_name])
POP_SELECTED = {"popularity_behavior_weights": pop_reports[best_pop]["weights"], "fallback": fallback_name,
                "fallback_weights": pop_reports[fallback_name]["weights"]}

prefix_full = prefix_events(val_q, val_ev, ("itemid", "event"))
prefix = prefix_full[prefix_full["distance"] < MAX_GROUPS]
rl_report, *_ = run(*repeat_last(prefix_full, qids, width=50), val_q, val_t, flt, fb_items, fb_scores)
sp_reports = {}
for wname, w in {"uniform": pop_reports[UNIFORM]["weights"], "best_pop_weights": pop_reports[best_pop]["weights"]}.items():
    r, *_ = run(*spop(prefix_full, qids, w, fb_items, fb_scores, width=50), val_q, val_t, flt, fb_items, fb_scores)
    r["weights"] = w
    sp_reports[f"spop_{wname}"] = r
best_spop = max(sp_reports, key=lambda n: sp_reports[n]["all"]["ndcg@20"])
display(flat({f"popularity ({best_pop})": pop_reports[best_pop], "repeat_last": rl_report, best_spop: sp_reports[best_spop]})[["hit@20", "mrr@20", "ndcg@20"]])
log(f"popularity/S-POP/Repeat-last xong: best_pop={best_pop}, fallback={fallback_name}, best_spop={best_spop}")

## 5. Họ kNN và SR (đúng lưới notebook 06)

In [ ]:
UNIFORM_W = {"view": 1, "addtocart": 1, "transaction": 1}
WEIGHT_OPTIONS = {"uniform": UNIFORM_W, "w_1_2_5": {"view": 1, "addtocart": 2, "transaction": 5},
                  "w_1_5_10": {"view": 1, "addtocart": 5, "transaction": 10}, "pop_best": POP_SELECTED["popularity_behavior_weights"]}
results, meta = {}, {}
t_knn = time.perf_counter()

def evaluate_model(name, model, fit_s, recommend):
    t0 = time.perf_counter()
    items, scores = recommend(model)
    q_s = time.perf_counter() - t0
    report, *_ = run(items, scores, val_q, val_t, flt, fb_items, fb_scores)
    require(report["short_lists"] == 0, f"{name}: có query thiếu item.")
    report.update(fit_seconds=round(fit_s, 2), query_seconds_total=round(q_s, 2), query_ms_per_query=round(1000 * q_s / len(val_q), 4))
    results[name] = report
    log(f"  {name:<42} ndcg@20={report['all']['ndcg@20']:.4f} fit={fit_s:.0f}s query={q_s:.0f}s")

itemknn_models = {}
def get_itemknn(m, wname):
    if (m, wname) not in itemknn_models:
        t0 = time.perf_counter()
        itemknn_models[(m, wname)] = (ItemKNN(m, WEIGHT_OPTIONS[wname]).fit(train_ev), time.perf_counter() - t0)
    return itemknn_models[(m, wname)]
for m in [50, 200]:
    for decay in [1.0, 0.6, 0.3]:
        model, fit_s = get_itemknn(m, "uniform")
        evaluate_model(f"itemknn_M{m}_decay{decay}_uniform", model, fit_s, lambda mdl, d=decay: mdl.recommend(prefix, qids, WIDTH, decay=d))
best_ik = max((n for n in results if n.startswith("itemknn")), key=lambda n: results[n]["all"]["ndcg@20"])
best_m, best_decay = int(best_ik.split("_M")[1].split("_")[0]), float(best_ik.split("decay")[1].split("_")[0])
itemknn_models = {k: v for k, v in itemknn_models.items() if k == (best_m, "uniform")}
for wname in [w for w in WEIGHT_OPTIONS if w != "uniform"]:
    model, fit_s = get_itemknn(best_m, wname)
    evaluate_model(f"itemknn_M{best_m}_decay{best_decay}_{wname}", model, fit_s, lambda mdl: mdl.recommend(prefix, qids, WIDTH, decay=best_decay))
    itemknn_models.pop((best_m, wname))
del itemknn_models
meta["itemknn"] = {"structure": {"n_neighbors": best_m, "decay": best_decay}}

def run_sknn(name, **params):
    t0 = time.perf_counter()
    model = SessionKNN(**params).fit(train_ev)
    evaluate_model(name, model, time.perf_counter() - t0, lambda mdl: mdl.recommend(prefix, qids, qtimes, WIDTH))

DAY = 86_400_000
structures = {}
for variant in ["sknn", "vsknn"]:
    for k in [50, 100, 200]:
        for sample in [500, 1000]:
            run_sknn(f"{variant}_k{k}_s{sample}_uniform", variant=variant, k=k, sample_size=sample)
    best = max((n for n in results if n.startswith(variant + "_")), key=lambda n: results[n]["all"]["ndcg@20"])
    structures[variant] = {"k": int(best.split("_k")[1].split("_")[0]), "sample_size": int(best.split("_s")[1].split("_")[0])}
base = structures["sknn"]
for l1 in [1.0, 3.0]:
    for l2 in [1, 7]:
        for l3 in [1.0, 3.0]:
            run_sknn(f"stan_k{base['k']}_s{base['sample_size']}_l1{l1}_l2{l2}d_l3{l3}_uniform", variant="stan",
                     k=base["k"], sample_size=base["sample_size"], lambda1=l1, lambda2_ms=l2 * DAY, lambda3=l3)
best_stan = max((n for n in results if n.startswith("stan_")), key=lambda n: results[n]["all"]["ndcg@20"])
l1 = float(best_stan.split("_l1")[1].split("_")[0]); l2 = int(best_stan.split("_l2")[1].split("d")[0])
l3 = float(best_stan.split("_l3")[1].split("_")[0])
structures["stan"] = {**base, "lambda1": l1, "lambda2_ms": l2 * DAY, "lambda3": l3}
for lidf in [0.5, 1.0, 2.0]:
    run_sknn(f"vstan_k{base['k']}_s{base['sample_size']}_idf{lidf}_uniform", variant="vstan", lambda_idf=lidf, **structures["stan"])
best_vstan = max((n for n in results if n.startswith("vstan_")), key=lambda n: results[n]["all"]["ndcg@20"])
structures["vstan"] = {**structures["stan"], "lambda_idf": float(best_vstan.split("_idf")[1].split("_")[0])}
for variant, params in structures.items():
    for wname in [w for w in WEIGHT_OPTIONS if w != "uniform"]:
        run_sknn(f"{variant}_best_{wname}", variant=variant, weights=WEIGHT_OPTIONS[wname], **params)
meta.update({v: {"structure": s} for v, s in structures.items()})

def run_sr(name, **params):
    t0 = time.perf_counter()
    model = SequentialRules(**params).fit(train_ev)
    evaluate_model(name, model, time.perf_counter() - t0, lambda mdl: mdl.recommend(prefix, qids, WIDTH))
for steps in [3, 10]:
    for n_rules in [100, 500]:
        run_sr(f"sr_steps{steps}_r{n_rules}_uniform", steps=steps, n_rules=n_rules)
best_sr = max((n for n in results if n.startswith("sr_")), key=lambda n: results[n]["all"]["ndcg@20"])
structures_sr = {"steps": int(best_sr.split("steps")[1].split("_")[0]), "n_rules": int(best_sr.split("_r")[1].split("_")[0])}
for wname in [w for w in WEIGHT_OPTIONS if w != "uniform"]:
    run_sr(f"sr_best_{wname}", weights=WEIGHT_OPTIONS[wname], **structures_sr)
meta["sr"] = {"structure": structures_sr}

family = lambda name: name.split("_")[0]
weight_of = lambda name: next(w for w in WEIGHT_OPTIONS if name.endswith("_" + w))
for fam in ["itemknn", "sknn", "vsknn", "stan", "vstan", "sr"]:
    best = max((n for n in results if family(n) == fam), key=lambda n: results[n]["all"]["ndcg@20"])
    meta[fam].update(best_run=best, weights=WEIGHT_OPTIONS[weight_of(best)], weight_name=weight_of(best))
require(len(results) == 51, f"Lưới kNN phải có 51 lần chạy như notebook 06, có {len(results)}.")
display(flat({f: results[m["best_run"]] for f, m in meta.items()})[["hit@20", "mrr@20", "ndcg@20"]])
log(f"kNN/SR xong ({(time.perf_counter() - t_knn) / 60:.0f} phút): " + ", ".join(f"{f}={m['best_run']}" for f, m in meta.items()))

## 6. GRU4Rec: cấu hình đã chọn, chọn số epoch trên validation 60 phút

In [ ]:
gcfg30 = read_json(ROOT / "models/gru4rec/config.json")
n_items = int(vocab["item_idx"].max()) + 1
idx_item = G.idx_to_itemid(vocab)
train_data, val_data = G.GroupData(train_ev, vocab), G.GroupData(val_ev, vocab)
windows = train_data.windows(MAX_LEN)
require(int((windows[:, 1] - 1).sum()) == len(samples["train"][0]), "Số bước train khác số query train.")

def gru_eval(model):
    items, scores = G.topn_items(model, G.query_states(model, val_data, val_q), idx_item, WIDTH)
    return run(items, scores, val_q, val_t, flt, fb_items, fb_scores)[0]

gru_reports = {}
def gru_on_epoch(epoch):
    gru_reports[epoch] = gru_eval(gru)
    log(f"  GRU4Rec epoch {epoch}: ndcg@20={gru_reports[epoch]['all']['ndcg@20']:.4f}")
    return gru_reports[epoch]["all"]["ndcg@20"]
torch.manual_seed(SEED)
gru = G.build_from_config(gcfg30, n_items=n_items)
t0 = time.perf_counter()
gru_hist = G.fit(gru, train_data, windows, GRU_MAX_EPOCHS, lr=gcfg30["lr"], batch_size=gcfg30["batch_size"], n_neg=gcfg30["n_neg"],
                 seed=SEED, on_epoch=gru_on_epoch, patience=PATIENCE, loss=gcfg30["loss"], bpreg=gcfg30["bpreg"],
                 elu=gcfg30["elu"], sample_alpha=gcfg30["sample_alpha"])
gru_fit_s = time.perf_counter() - t0
gru_best = max(gru_reports, key=lambda e: gru_reports[e]["all"]["ndcg@20"])
require(abs(gru_eval(gru)["all"]["ndcg@20"] - gru_reports[gru_best]["all"]["ndcg@20"]) < 1e-12, "Không khôi phục được trạng thái epoch tốt nhất.")
gcfg60 = {**gcfg30, "n_items": n_items, "epochs": gru_best, "vocabulary": "train (60-minute sessions)"}
(MODEL_DIR / "gru4rec").mkdir(parents=True, exist_ok=True)
torch.save(gru.state_dict(), MODEL_DIR / "gru4rec/model.pt")
write_json(MODEL_DIR / "gru4rec/config.json", gcfg60)
gru_lists = G.topn_items(gru, G.query_states(gru, val_data, val_q), idx_item, WIDTH)
del train_data, windows
log(f"GRU4Rec xong ({gru_fit_s / 60:.0f} phút): epoch tốt nhất {gru_best}, ndcg@20={gru_reports[gru_best]['all']['ndcg@20']:.4f}")

## 7. SR-GNN: cấu hình đã chọn, chọn số epoch trên validation 60 phút

In [ ]:
scfg30 = read_json(ROOT / "models/srgnn/config.json")
train_q, train_t = samples["train"]
train_graphs = S.GraphData(prefix_events(train_q, train_ev, ("itemid", "event"), max_groups=MAX_GROUPS), train_q["query_id"].to_numpy(), vocab)
pos_ptr, pos_val = S.target_csr(train_t, train_q["query_id"].to_numpy(), vocab)
require(len(train_graphs) == len(train_q) and (np.diff(pos_ptr) > 0).all(), "Mỗi query train phải có target trong vocabulary.")
neg_probs, log_q = S.negative_distribution(pos_val, n_items, scfg30["sample_alpha"])
val_graphs = S.GraphData(prefix, qids, vocab)

def srgnn_eval(model):
    items, scores = G.topn_items(model, S.session_vectors(model, val_graphs), idx_item, WIDTH)
    return run(items, scores, val_q, val_t, flt, fb_items, fb_scores)[0]

sr_reports = {}
def sr_on_epoch(epoch):
    sr_reports[epoch] = srgnn_eval(srgnn)
    log(f"  SR-GNN epoch {epoch}: ndcg@20={sr_reports[epoch]['all']['ndcg@20']:.4f}")
    return sr_reports[epoch]["all"]["ndcg@20"]
torch.manual_seed(SEED)
srgnn = S.SRGNN(n_items, dim=scfg30["dim"], steps=scfg30["steps"], dropout=scfg30["dropout"])
t0 = time.perf_counter()
sr_hist = S.fit(srgnn, train_graphs, pos_ptr, pos_val, neg_probs, log_q, SRGNN_MAX_EPOCHS, lr=scfg30["lr"],
                batch_size=scfg30["batch_size"], n_neg=scfg30["n_neg"], seed=SEED, on_epoch=sr_on_epoch, patience=PATIENCE)
sr_fit_s = time.perf_counter() - t0
sr_best = max(sr_reports, key=lambda e: sr_reports[e]["all"]["ndcg@20"])
require(abs(srgnn_eval(srgnn)["all"]["ndcg@20"] - sr_reports[sr_best]["all"]["ndcg@20"]) < 1e-12, "Không khôi phục được trạng thái epoch tốt nhất.")
scfg60 = {**scfg30, "n_items": n_items, "epochs": sr_best, "vocabulary": "train (60-minute sessions)"}
(MODEL_DIR / "srgnn").mkdir(parents=True, exist_ok=True)
torch.save(srgnn.state_dict(), MODEL_DIR / "srgnn/model.pt")
write_json(MODEL_DIR / "srgnn/config.json", scfg60)
del train_graphs, pos_ptr, pos_val
log(f"SR-GNN xong ({sr_fit_s / 60:.0f} phút): epoch tốt nhất {sr_best}, ndcg@20={sr_reports[sr_best]['all']['ndcg@20']:.4f}")

## 8. Reranker kiểu HySAR (như notebook 09)

In [ ]:
from rr.evaluation import fit_neighbour_model, recommend_neighbour
t0 = time.perf_counter()
knn_family = max(meta, key=lambda f: results[meta[f]["best_run"]]["all"]["ndcg@20"])
knn_lists = recommend_neighbour(fit_neighbour_model(knn_family, meta, train_ev), knn_family, meta, prefix, qids, qtimes, WIDTH)
base_reports = {knn_family: run(*knn_lists, val_q, val_t, flt, fb_items, fb_scores)[0],
                "gru4rec": run(*gru_lists, val_q, val_t, flt, fb_items, fb_scores)[0]}
require(abs(base_reports[knn_family]["all"]["ndcg@20"] - results[meta[knn_family]["best_run"]]["all"]["ndcg@20"]) < 1e-9,
        "Fit lại kNN không khớp lưới.")
sources = {"knn": knn_lists, "gru": gru_lists}
table = add_features(candidate_table(qids, sources, prefix, top_n=TOP_N), val_q, prefix, train_ev, history)
table["y"] = label(table, val_t)
FEATURES = feature_columns(table, sources)
coverage = float(table["y"].sum() / len(val_t))
folds = session_folds(val_q, 2, seed=SEED)
fold_of_row = pd.Series(folds, index=qids).reindex(table["query_id"].to_numpy()).to_numpy()
has_pos = table.groupby("query_id")["y"].transform("max").to_numpy() > 0

def lgb_data(mask):
    part = table[mask]
    return lgb.Dataset(part[FEATURES], part["y"], group=part.groupby("query_id", sort=False).size().to_numpy(), free_raw_data=False)

rcfg30 = read_json(ROOT / "reports/eval/reranker_validation.json")["cv"]
GRID = [{k: v["params"][k] for k in ["num_leaves", "min_data_in_leaf", "learning_rate"]} for v in rcfg30.values()]
cv = {}
for gi, params in enumerate(GRID):
    full = {"objective": "lambdarank", "metric": "ndcg", "eval_at": [20], "lambdarank_truncation_level": 20,
            "feature_fraction": 0.9, "bagging_fraction": 0.8, "bagging_freq": 1, "seed": SEED,
            "deterministic": True, "num_threads": 4, "verbose": -1, **params}
    oof, best_iters = np.zeros(len(table)), []
    for k in [0, 1]:
        tr, va = (fold_of_row != k) & has_pos, (fold_of_row == k) & has_pos
        booster = lgb.train(full, lgb_data(tr), num_boost_round=1000, valid_sets=[lgb_data(va)],
                            callbacks=[lgb.early_stopping(50, verbose=False)])
        best_iters.append(booster.best_iteration)
        held = fold_of_row == k
        oof[held] = booster.predict(table.loc[held, FEATURES], num_iteration=booster.best_iteration)
    report, *_ = run(*ranked_lists(table, oof, qids, WIDTH), val_q, val_t, flt, fb_items, fb_scores)
    cv[f"grid{gi}"] = {"params": full, "best_iterations": best_iters, "report": report}
    log(f"  reranker grid{gi}: OOF ndcg@20={report['all']['ndcg@20']:.4f} iters={best_iters}")
best_cv = max(cv, key=lambda n: cv[n]["report"]["all"]["ndcg@20"])
n_rounds = int(np.mean(cv[best_cv]["best_iterations"]))
final = lgb.train(cv[best_cv]["params"], lgb_data(has_pos), num_boost_round=n_rounds)
(MODEL_DIR / "reranker").mkdir(parents=True, exist_ok=True)
(MODEL_DIR / "reranker/model.txt").write_bytes(final.model_to_string().encode("utf-8"))  # LightGBM cannot open non-ASCII paths
rcfg60 = {"features": FEATURES, "sources": {"knn": knn_family, "gru": gcfg60["name"]}, "top_n": TOP_N, "params": cv[best_cv]["params"],
          "num_boost_round": n_rounds, "trained_on": "validation queries of 60-minute sessions (base models fitted on 60-minute train)"}
write_json(MODEL_DIR / "reranker/config.json", rcfg60)
importance = pd.Series(final.feature_importance("gain"), index=FEATURES).sort_values(ascending=False)
del table
log(f"reranker xong ({(time.perf_counter() - t0) / 60:.0f} phút): {best_cv}, OOF ndcg@20={cv[best_cv]['report']['all']['ndcg@20']:.4f}")

## 9. So sánh với validation 30 phút và lưu báo cáo

Hai cột dùng **hai bộ query khác nhau** (validation theo từng định nghĩa phiên), mỗi cột là cấu hình tốt nhất sau khi tune trên chính nó; chỉ đọc như mô tả. So sánh trên cùng bộ query nằm ở notebook 14 (test).

In [ ]:
ev30 = {n: read_json(ROOT / f"reports/eval/{n}_validation.json") for n in ["popularity", "spop", "repeat_last", "knn", "gru4rec", "srgnn", "reranker"]}
best30 = {"popularity": ev30["popularity"]["grid"][ev30["popularity"]["best"]], "spop": ev30["spop"]["runs"][ev30["spop"]["best"]],
          "repeat_last": ev30["repeat_last"]["report"],
          **{f: ev30["knn"]["runs"][m["best_run"]] for f, m in ev30["knn"]["selected"].items()},
          "gru4rec": ev30["gru4rec"]["runs"][ev30["gru4rec"]["selected"]["name"]]["report"],
          "srgnn": ev30["srgnn"]["runs"][ev30["srgnn"]["selected"]["name"]]["report"],
          "reranker": ev30["reranker"]["cv"][ev30["reranker"]["selected"]]["report"]}
best60 = {"popularity": pop_reports[best_pop], "spop": sp_reports[best_spop], "repeat_last": rl_report,
          **{f: results[m["best_run"]] for f, m in meta.items()}, "gru4rec": gru_reports[gru_best], "srgnn": sr_reports[sr_best],
          "reranker": cv[best_cv]["report"]}
compare = pd.DataFrame({n: {"30 phút": best30[n]["all"]["ndcg@20"], "60 phút": best60[n]["all"]["ndcg@20"],
                            "bài toán 2 (30)": best30[n].get("strong_behavior_targets", {}).get("ndcg@20", np.nan),
                            "bài toán 2 (60)": best60[n].get("strong_behavior_targets", {}).get("ndcg@20", np.nan)} for n in best30}).T
compare["Δ"] = compare["60 phút"] - compare["30 phút"]
display(compare.style.format("{:.4f}"))
selection = pd.DataFrame({"30 phút": {"popularity": ev30["popularity"]["best"], "spop": ev30["spop"]["best"],
                                     **{f: m["best_run"] for f, m in ev30["knn"]["selected"].items()},
                                     "gru4rec_epochs": ev30["gru4rec"]["selected"]["epochs"], "srgnn_epochs": ev30["srgnn"]["selected"]["epochs"],
                                     "reranker": ev30["reranker"]["selected"]},
                          "60 phút": {"popularity": best_pop, "spop": best_spop, **{f: m["best_run"] for f, m in meta.items()},
                                     "gru4rec_epochs": gru_best, "srgnn_epochs": sr_best, "reranker": best_cv}})
display(selection)

require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
for d in ["gru4rec/model.pt", "gru4rec/config.json", "srgnn/model.pt", "srgnn/config.json", "reranker/model.txt", "reranker/config.json"]:
    artifacts[f"models/session60/{d}"] = fingerprint(MODEL_DIR / d)
write_json(OUT / "validation.json", {
    "generated_at_utc": utc_now(), **environment(), "torch_version": torch.__version__, "lightgbm_version": lgb.__version__,
    "notebook": "notebooks/13_phien_60_phut_tune_validation.ipynb",
    "protocol": {"path": rel(PROTOCOL_PATH, ROOT), "version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
    "variant": {"inactivity_threshold_ms": THRESHOLD_MS, "cutoff_ms": CUTOFFS, "status": "secondary report; locked protocol stays 30 minutes"},
    "inputs": inputs, "artifacts": artifacts, "split_summary": SUMMARY, "samples": data_stats,
    "vocabulary": {"train": int(len(vocab)), "train_validation": int(len(vocab_tv))},
    "popularity": {"grid": pop_reports, "uniform": UNIFORM, "best": best_pop, "selected": POP_SELECTED},
    "spop": {"runs": sp_reports, "best": best_spop}, "repeat_last": {"report": rl_report},
    "knn": {"weight_options": WEIGHT_OPTIONS, "runs": results, "selected": meta},
    "gru4rec": {"history": gru_hist, "reports_by_epoch": {str(e): r for e, r in gru_reports.items()}, "best_epoch": gru_best,
                "fit_seconds": round(gru_fit_s, 1), "config": gcfg60},
    "srgnn": {"history": sr_hist, "reports_by_epoch": {str(e): r for e, r in sr_reports.items()}, "best_epoch": sr_best,
              "fit_seconds": round(sr_fit_s, 1), "config": scfg60},
    "reranker": {"base_models": base_reports, "candidate_target_coverage": coverage, "cv": cv, "selected": best_cv, "config": rcfg60,
                 "feature_importance_gain": importance.to_dict()},
    "compare_with_30min_validation": {"note": "different query sets; best configuration of each threshold",
                                      "ndcg@20": compare.to_dict(orient="index"), "selection": selection.to_dict()},
    "status": "PASS"})
log("xong, báo cáo: " + rel(OUT / "validation.json", ROOT))

## Kết luận (validation 60 phút)

- Cấu hình được chọn gần như trùng với 30 phút:
  - popularity: v1 c2 b1;
  - itemknn, sknn, vsknn, stan, sr: giữ nguyên cấu hình;
  - vstan: idf 2.0 (30 phút chọn 0.5);
  - GRU4Rec 2 epoch, SR-GNN 5 epoch;
  - reranker: grid0 (30 phút chọn grid1).
- NDCG@20 validation của các mô hình chính tăng khoảng +0,002 đến +0,006. Đây là hai bộ query khác nhau nên không so trực tiếp được; so sánh trên cùng bộ query nằm ở notebook 14.
